# Incremental amplitude-damping sweep for new PI codes

This notebook only evaluates the newly added codes over five log-spaced points between $p=10^{-3}$ and $p=10^{-2}$. It writes separate local/global CSV files so the older all-code sweeps do not need to be rerun.

In [2]:
from pathlib import Path
import csv
import gc
import os
import sys
import time

import numpy as np

# Make imports robust whether Jupyter starts in the repo root or in python/notebooks.
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "python" / "codes").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Could not locate the repository root containing python/codes.")

# Keep package/cache paths local to this repo/session.
sys.path.insert(0, str(repo_root / "python"))
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".matplotlib-cache"))
os.environ["MOSEKLM_LICENSE_FILE"] = str(repo_root / "mosek" / "mosek.lic")

from codes.codewords import (
    kubischta_teixeira_11_piqs,
    pollatsek_ruskai_7_piqs,
)
from codes.noisemodel import noisemodel
from codes.optimisation import no_recovery, optimise

DATA_DIR = repo_root / "datas"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"repo_root = {repo_root}")
print(f"data_dir = {DATA_DIR}")

Matplotlib is building the font cache; this may take a moment.


repo_root = /Users/46668993/Desktop/qer
data_dir = /Users/46668993/Desktop/qer/datas


In [1]:
# Five points from p=1e-3 to p=1e-2. Here p = gamma * dt.
dt = 1.0
p_vals = np.logspace(-3, -2, 5)
gamma_vals = p_vals / dt

method = "choi"
solver = "mosek"

# Only the three new code entries.
rho_pr7_plus, l0_pr7_plus, l1_pr7_plus = pollatsek_ruskai_7_piqs(epsilon=1, return_qutip=True)
rho_pr7_minus, l0_pr7_minus, l1_pr7_minus = pollatsek_ruskai_7_piqs(epsilon=-1, return_qutip=True)
rho_kt11, l0_kt11, l1_kt11 = kubischta_teixeira_11_piqs(return_qutip=True)

code_specs = [
    {
        "label": "Pollatsek-Ruskai 7 (+)",
        "prefix": "pr7_plus",
        "N": 7,
        "rho": rho_pr7_plus,
        "l0": l0_pr7_plus,
        "l1": l1_pr7_plus,
    },
    {
        "label": "Pollatsek-Ruskai 7 (-)",
        "prefix": "pr7_minus",
        "N": 7,
        "rho": rho_pr7_minus,
        "l0": l0_pr7_minus,
        "l1": l1_pr7_minus,
    },
    {
        "label": "Kubischta-Teixeira 11",
        "prefix": "kt11",
        "N": 11,
        "rho": rho_kt11,
        "l0": l0_kt11,
        "l1": l1_kt11,
    },
]

for spec in code_specs:
    print(f"{spec['label']}: N={spec['N']}, rho shape={spec['rho'].shape}")
print("p values:", p_vals)

NameError: name 'np' is not defined

In [ ]:
def run_incremental_amp_damp_sweep(noise_name, csv_filename):
    results = {}
    for spec in code_specs:
        prefix = spec["prefix"]
        results[f"{prefix}_no_recovery"] = []
        results[f"{prefix}_optimal"] = []

    for gamma, p in zip(gamma_vals, p_vals):
        print(f"\n{noise_name}: p={p:.3e}, gamma={gamma:.3e}")
        for spec in code_specs:
            prefix = spec["prefix"]
            label = spec["label"]
            t0 = time.time()
            try:
                channel = noisemodel(noise_name, spec["N"], gamma, dt, method)
                fid_no_recovery = no_recovery(spec["rho"], channel)
                fid_optimal = optimise(spec["l0"], spec["l1"], channel, solver=solver)

                inf_no_recovery = abs(1.0 - float(fid_no_recovery))
                inf_optimal = abs(1.0 - float(fid_optimal))
                print(
                    f"  {label}: no recovery={inf_no_recovery:.6e}, "
                    f"optimal={inf_optimal:.6e}, time={time.time() - t0:.1f}s"
                )
            except Exception as exc:
                print(f"  ERROR {label}: {exc}")
                inf_no_recovery = np.nan
                inf_optimal = np.nan
            finally:
                gc.collect()

            results[f"{prefix}_no_recovery"].append(inf_no_recovery)
            results[f"{prefix}_optimal"].append(inf_optimal)

    csv_path = DATA_DIR / csv_filename
    result_keys = list(results.keys())
    with csv_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["gamma_dt", *result_keys])
        for i, p in enumerate(p_vals):
            writer.writerow([p, *[results[key][i] for key in result_keys]])

    print(f"\nSaved CSV to: {csv_path}")
    return csv_path, results

In [ ]:
local_csv_path, local_results = run_incremental_amp_damp_sweep(
    "local symmetric amplitude damping",
    "local_amp_damp_new_codes_results.csv",
)

In [ ]:
global_csv_path, global_results = run_incremental_amp_damp_sweep(
    "global symmetric amplitude damping",
    "global_amp_damp_new_codes_results.csv",)

In [ ]:
print("Local CSV:", local_csv_path)
print("Global CSV:", global_csv_path)